# EverFlavor AI - Climate and Season: Carbon Footprint and What Is in Season

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: planning sketch.** The sections, sources, output tables and functions below are the plan.
> The functions are stubs (`NotImplementedError`) and nothing here has been run or tested yet.
> "Run all" is safe: it only runs the setup and prints the plan's status.

---

## Purpose

Two more things a person may care about when choosing a dish:

- **Climate:** roughly how much greenhouse gas the ingredients cause (kg CO2-equivalent), so the Chef Agent
  can mention a lower-impact swap (lentils for beef) when someone asks.
- **Season:** which fruits and vegetables are in season this month where the person lives, so suggestions
  favor fresh, cheaper produce.

**Honesty rules:** without ingredient amounts, a dish gets an **impact class** from its main ingredients
(for example "contains beef: high impact"), never a precise kg CO2e per serving. Footprints are global or
French averages, not the person's farm or store. Seasons vary by region and weather: "usually in season".

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | `data/interim/recipes_all.parquet` (ingredient lists) | Notebook 01 (section 5.11) |
| Input | Our World in Data: GHG emissions per kg of food (Poore & Nemecek 2018), CC BY | Section 2, small CSV download |
| Input | AGRIBALYSE 3.2 (ADEME, France): 2,516 foods, Etalab Open License | Section 2, CSV download |
| Input | A seasonal produce calendar by month and region (USDA SNAP-Ed guide; license to confirm) | Section 3 |
| Output | `data/processed/ingredient_footprint.csv` (kg CO2e per kg, source, match) | Notebook 08 (section 2) |
| Output | `data/processed/recipe_impact.parquet` (impact class per recipe) | Notebook 08 (section 4) |
| Output | `data/processed/produce_seasons.csv` (item, region, months) | Notebook 08 (section 3) |

## Contents

1. Setup
2. Carbon Footprint per Ingredient
3. What Is in Season
4. Impact Class per Recipe
5. Team Decisions and Next Steps

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q requests pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "calories", "charts", "checks", "cooking", "cuisine", "diets",
                      "environment", "features", "flags", "freshness", "ingredients", "knowledge", "llm",
                      "nutrition", "nutrition_quality", "parsing", "pipeline", "progress", "recommend",
                      "reporting", "review", "safety", "sources", "stores", "variants"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
FOOTPRINT_DIR = Path("data/raw/footprint")
for folder in [FOOTPRINT_DIR, Path("data/processed")]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

Runtime        : Local Jupyter (VS Code / Antigravity / JupyterLab)
Python         : 3.12.10
Project folder : ~\Downloads\jupiter-exploration\The-EverFlavor-AI
Notebook 01 outputs found.


---

## 2. Carbon Footprint per Ingredient

| Source | What it has | License | Use |
|---|---|---|---|
| [Our World in Data](https://ourworldindata.org/grapher/ghg-per-kg-poore) (Poore & Nemecek 2018, *Science*) | Global average kg CO2e per kg for about 40 foods (beef, lamb, cheese, rice, lentils ...) from ~38,700 farms in 119 countries | CC BY ("completely open access under the Creative Commons BY license") | Main source: broad food groups, worldwide |
| [AGRIBALYSE 3.2](https://doc.agribalyse.fr/documentation-en/agribalyse-data/data-access) (ADEME) | 2,516 finished foods sold in France, climate change per kg and by life-cycle stage | Etalab Open License (reuse, also commercial, with the source and date credited) | Finer detail for prepared foods; French averages |

**Matching:** each of our ingredient names is matched to a food group ("ground beef" -> Beef (beef herd),
"cheddar cheese" -> Cheese), like the USDA matches of notebook 01, with a hand check of the top matches.

In [3]:
OWID_GHG_URL = "https://ourworldindata.org/grapher/ghg-per-kg-poore.csv?v=1&csvType=full&useColumnShortNames=false"
AGRIBALYSE_PAGE = "https://doc.agribalyse.fr/documentation-en/agribalyse-data/data-access"


def footprint_table(folder: Path, refresh: bool = False) -> pd.DataFrame:
    """Download Our World in Data's GHG per kg table (cached): food, kg_co2e_per_kg, source."""
    raise NotImplementedError("sketch: footprint download")


def match_footprints(ingredients: list[str], footprints: pd.DataFrame) -> pd.DataFrame:
    """Match ingredient names to footprint foods: ingredient, food, kg_co2e_per_kg, match (exact / group / hand-check)."""
    raise NotImplementedError("sketch: footprint matching")

---

## 3. What Is in Season

**Candidate source:** the [USDA SNAP-Ed Seasonal Produce Guide](https://wicworks.fns.usda.gov/resources/snap-ed-seasonal-produce-guide)
(US, by season). It is a government resource, but its license is not stated: **to confirm** before use.
Regions outside the US need their own calendars (team decision: which regions first, notebook 03's pilot).

**Planned table** (`produce_seasons.csv`): `item`, `region`, `months` (list), `source`.

In [4]:
def produce_seasons(region: str) -> pd.DataFrame:
    """Months each fruit and vegetable is usually in season in `region`."""
    raise NotImplementedError("sketch: seasonal calendar")

---

## 4. Impact Class per Recipe

Without amounts, a recipe's class comes from its highest-impact main ingredient: **high** (beef, lamb,
cheese, prawns from farms), **medium** (pork, poultry, eggs, rice), **low** (legumes, grains, most
vegetables and fruit). Thresholds in kg CO2e per kg are a team decision; they must come from the source
table, not from guesses.

In [5]:
IMPACT_CLASSES = {"high": 20.0, "medium": 4.0}   # kg CO2e per kg at or above; placeholder until the team decides


def recipe_impact(recipes: pd.DataFrame, footprints: pd.DataFrame) -> pd.DataFrame:
    """recipe_id, impact_class, driver (the ingredient that sets the class)."""
    raise NotImplementedError("sketch: impact class")

---

## 5. Team Decisions and Next Steps

1. **Sources:** Our World in Data first (small, CC BY); AGRIBALYSE for prepared foods; confirm the USDA
   seasonal guide's license or pick another calendar.
2. **Impact thresholds** for high / medium / low.
3. **Regions** for the seasonal calendar, following notebook 03's pilot.
4. **Wording for the agent:** "a lower-impact choice" and "usually in season", never exact emissions per dish.

**Build order:** footprints (section 2), impact class (section 4), then seasons (section 3) once a calendar
source is confirmed.

---

## Plan status

In [6]:
PLAN = pd.DataFrame([
    ("Plan, sources, output tables", "written; licenses read 2026-10-05"),
    ("2 Footprint per ingredient", "stub - Our World in Data CSV (CC BY) ready to download"),
    ("3 What is in season", "stub - calendar source and license to confirm"),
    ("4 Impact class per recipe", "stub - needs team thresholds"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))

                        step                                                 status
Plan, sources, output tables                      written; licenses read 2026-10-05
  2 Footprint per ingredient stub - Our World in Data CSV (CC BY) ready to download
         3 What is in season          stub - calendar source and license to confirm
   4 Impact class per recipe                           stub - needs team thresholds
